# Market simulator demo

Demonstrates `market_simulator.py` with PnL accounting and per-order partial-fill attribution.

In [ ]:
import json
from market_simulator import MarketSimulator
from cobr_signal import COBRSignal
from sample_real_data_adapter import parse_csv_trades

sim = MarketSimulator(starting_cash=100000.0)
sig = COBRSignal()

# create synthetic trades: price oscillates
ticks = []
prices = [100.0, 99.9, 100.2, 100.1, 100.5, 100.3, 100.0]
for i, p in enumerate(prices):
    prev_price = prices[i - 1] if i else p
    tick = {"ts": i + 0.0, "price": p, "prev_price": prev_price, "qty": 5.0, "side": "buy" if i % 2 == 0 else "sell"}
    ticks.append(tick)
    signal = sig.generate(tick)
    print("signal", signal)

for t in ticks:
    sim.ingest_trade(t)

# place two limit orders that can partially fill against synthetic ticks
oid1 = sim.place_limit(side="buy", price=100.1, qty=8.0, ts=0.001, client_id="client_A")
oid2 = sim.place_limit(side="sell", price=100.3, qty=6.0, ts=0.002, client_id="client_B")

print("Orders:")
for order in sim.list_orders():
    print(order.order_id, order.side, order.price, "orig_qty=", order.qty, "remaining=", order.remaining, "status=", order.status)

print("\nFills:")
for fill in sim.list_fills():
    print(fill.order_id, fill.side, fill.price, fill.qty, getattr(fill, "realized_pnl", None))

print("\nPnL snapshot:")
print(json.dumps(sim.get_pnl(), indent=2))
